# 🎓 第4章 第1课：七大模型 —— 从三因子到 q-factor

**本节目标**：把第3章的七个因子组合成七个主流多因子模型，看清它们之间的关系。

对应原书第4章。模型不是“因子越多越好”，而是**用尽量少的因子解释尽量多的收益差异**。

> ⚠️ 数据声明：本仓库的估值/ROE/投资为教学合成面板；模型比较的数值仅演示流程，不代表真实 A 股结论。

In [1]:
import sys
from pathlib import Path

def _find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'src' / 'factor_zero').exists():
            return p
    raise FileNotFoundError('找不到仓库根目录，请从仓库内启动 notebook')
ROOT = _find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from factor_zero.data import load_ashare_panel
from factor_zero.factors import build_factor_returns
from factor_zero.models import (
    MODEL_DEFINITIONS, model_factor_frame, build_test_assets,
    evaluate_model, compare_models,
)
from factor_zero.regression import fama_macbeth, time_series_betas
from factor_zero.statistics import factor_summary

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans']
print('✅ 环境就绪，开始学习！')

✅ 环境就绪，开始学习！


## 1) 七个模型的定义

| 模型 | 因子 | 一句话 |
|---|---|---|
| FF3 | MKT + SMB + HML | 市场、规模、价值 |
| Carhart4 | FF3 + WML | 加动量 |
| NovyMarx4 | MKT + HML + WML + RMW | 市场、价值、动量、盈利 |
| FF5 | MKT + SMB + HML + RMW + CMA | 加盈利、投资 |
| HXZ4 | MKT + SMB + RMW + CMA | q-factor：市场、规模、盈利、投资 |
| SY4 | MKT + SMB + RMW + WML | 近似：RMW/WML 代理 MGMT/PERF |
| DHS3 | MKT + HML + WML | 近似：HML/WML 代理 FIN/PEAD |

> SY4 和 DHS3 的真实因子（MGMT/PERF、FIN/PEAD）本仓库数据无法精确构造，
> 这里用现有因子做“近似代理”，只用于演示模型比较流程，**不代表原模型**。

In [2]:
panel = load_ashare_panel()
factors = build_factor_returns(panel, n_groups=10)
model_table = pd.DataFrame([
    {'模型': name, '因子数': len(spec['factors']),
     '因子': ' + '.join(spec['factors']), '说明': spec['note']}
    for name, spec in MODEL_DEFINITIONS.items()
]).set_index('模型')
model_table

,因子数,因子,说明
模型,,,
FF3,3,MKT + SMB + HML,Fama-French 三因子
Carhart4,4,MKT + SMB + HML + WML,FF3 + 动量
NovyMarx4,4,MKT + HML + WML + RMW,市场 + 价值 + 动量 + 盈利
FF5,5,MKT + SMB + HML + RMW + CMA,Fama-French 五因子
HXZ4,4,MKT + SMB + RMW + CMA,q-factor：市场 + 规模 + 盈利 + 投资
SY4,4,MKT + SMB + RMW + WML,近似：RMW/WML 代理 MGMT/PERF
DHS3,3,MKT + HML + WML,近似：HML/WML 代理 FIN/PEAD


## 2) 模型之间的关系

- **FF3 ⊂ Carhart4**：Carhart4 = FF3 + 动量；
- **FF3 ⊂ FF5**：FF5 = FF3 + 盈利 + 投资；
- **HXZ4** 和 FF5 共享 MKT/SMB/RMW/CMA，但 HXZ4 不含 HML；
- **NovyMarx4** 不含 SMB，强调价值、动量、盈利；
- **SY4/DHS3** 在这里是近似代理，不是原版模型。

> 嵌套模型可以用 GRS/α 检验比较；非嵌套模型还可以看均值-方差张成。

In [3]:
rows = []
for name in MODEL_DEFINITIONS:
    mf = model_factor_frame(factors, name)
    rows.append({'模型': name, '因子数': mf.shape[1],
                 '有效月份': int(mf.dropna().shape[0]),
                 '因子': ' + '.join(mf.columns)})
pd.DataFrame(rows).set_index('模型')

,因子数,有效月份,因子
模型,,,
FF3,3,82,MKT + SMB + HML
Carhart4,4,70,MKT + SMB + HML + WML
NovyMarx4,4,70,MKT + HML + WML + RMW
FF5,5,82,MKT + SMB + HML + RMW + CMA
HXZ4,4,82,MKT + SMB + RMW + CMA
SY4,4,70,MKT + SMB + RMW + WML
DHS3,3,70,MKT + HML + WML


## 3) 先看一眼：不同模型的因子收益

同一组因子，组合方式不同，模型的“性格”也不同。
下一课会用 GRS 正式比较；这里先建立直觉。

In [4]:
summary = {}
for name in ['FF3', 'Carhart4', 'FF5', 'HXZ4']:
    mf = model_factor_frame(factors, name)
    # 用等权因子组合代表模型（仅作直觉展示）
    summary[name] = mf.mean(axis=1)
pd.DataFrame({name: factor_summary(pd.Series(vals).dropna())
              for name, vals in summary.items()}).T.round(4)

,mean_ann,vol_ann,sharpe,t_stat,max_drawdown,win_rate
FF3,0.0471,0.1274,0.3699,0.9716,-0.2795,0.4634
Carhart4,0.0783,0.1062,0.7369,1.7535,-0.1841,0.5976
FF5,0.0544,0.0714,0.7623,1.7325,-0.0726,0.5976
HXZ4,0.1201,0.0946,1.2684,2.7688,-0.1200,0.6220


## ✋ 试一试

1. 把 FF5 的 CMA 去掉，和 FF3+WML 比，哪个平均 |α| 更低？（下一课验证）
2. 把 HXZ4 的 RMW 换成 HML，模型变成什么？
3. 如果只能用 3 个因子，你会选哪 3 个？为什么？

## ✅ 本节小结

- 七大模型是七个因子集的不同组合；
- 嵌套关系：FF3 ⊂ Carhart4、FF3 ⊂ FF5；
- 模型选择不是“越多越好”，要看解释力、显著性和简约性；
- SY4/DHS3 在本仓库是近似代理，不能当作原模型结论。

> 下一课：用 GRS 检验正式比较七个模型。